теперь приступим к дообучению, устанавливаем библиотеку transformers для работы с моделями, импортируем автотокенезатор (у каждой модели свой токенизатор и просто так свой засунуть не получится,можно прописать его в ручуную,но если использовать его то токенизатор подберется автоматически к модели), берем модель (я выбрал руберта от МФТИ который обучен на русской википедии,он хорош по отзовам и немного весит,что позволит его быстро дообучить), затем для проверки можем использовать рандомный отзыв и увидеть как токенизатор его разбивает

In [1]:
%pip install transformers
%pip install torch torchvision torchaudio
from transformers import AutoTokenizer

MODEL_NAME = 'DeepPavlov/rubert-base-cased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

sample_text = "Этот фильм вообще не понравился,полный отстой!"
tokens = tokenizer.tokenize(sample_text)

print(f"Оригинальный текст: {sample_text}")
print(f"Токены: {tokens}")


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Оригинальный текст: Этот фильм вообще не понравился,полный отстой!
Токены: ['Этот', 'фильм', 'вообще', 'не', 'понравился', ',', 'полный', 'отстой', '!']


Загружаем все наши данные и объединяем для лучшего обучения

In [ ]:
import pandas as pd
dt_train = pd.read_json("train.jsonl", lines=True)
dt_test = pd.read_json("test.jsonl", lines=True)
dt_val = pd.read_json("validation.jsonl", lines=True)

Можем посмотреть на encoding и decoding,видим,что все работает отлично (101 и 102 это обозначени для начала и конца предложения)

In [3]:
encoded_text = tokenizer.encode(sample_text)
print(f"encode: {encoded_text}")


decoded_text = tokenizer.decode(encoded_text)
print(f"decode: {decoded_text}")

encode: [101, 11514, 7142, 17050, 1699, 58171, 128, 21228, 110950, 106, 102]
decode: [CLS] Этот фильм вообще не понравился, полный отстой! [SEP]


Импортируем numpy и tqdm, заводим коробку для измерений в которцю мы будем класть длину отзывов, создаем текст для измерения длин текстов, мы просим токенизатор перевести наш текст в цифры(сразу ставим огроничение в 512 так как у берта это техническое ограничение в количестве токенов,truncation=True для обрезки текстов длинее 512 токенов), затем считаем сколько токенов получилось и кидаем в нашу коробку, затем мы не просто берем максимальный размер одного текста,мы считаем по всем и выводим значение при котором 95% текста уместится в такую длину,откидывая 5% текстов аномального размера, ставим ограничение в 512 и сверху нашего значения на всякий случай накидываем еще 10, у нас получилось,что в основном отзывы перешагивают порог в 512 токенов

In [4]:
import numpy as np
from tqdm import tqdm

print("Анализируем длину текстов")
token_lens = []


for txt in tqdm(dt_train['text']):
    
    tokens = tokenizer.encode(txt, max_length=512, truncation=True)
    token_lens.append(len(tokens))

optimal_max_len = int(np.percentile(token_lens, 95))
print(f"95% текстов имеют длину не более: {optimal_max_len} токенов.")

MAX_LEN = min(optimal_max_len + 10, 512) 
print(f"MAX_LEN = {MAX_LEN}")

Анализируем длину текстов


100%|██████████| 10500/10500 [00:07<00:00, 1432.87it/s]

95% текстов имеют длину не более: 512 токенов.
MAX_LEN = 512


Импортируем торч и достаем dataset и dataloader (первый просто шаблон для того чтобы торч мог читать данные, второй нужен для создания батчей), затем создаем класс ReviewDataset который наследуется от базового класса из pytorch, создаем метод init в котором мы передаем все наши данные (текст,лейблы,токенизатор и максималную длину текста), создаем функцию для определения количества отзовов (нужно для того что бы торч понимал на сколько бачей разбивать) и функцию для каждого нашего отзыва которая будет по сути циклом прогоняться каждый раз, мы обращаемся к таблице и вытаскиваем наш отзыв прописываем str на всякий случай если отзыв будет пустым, чтобы токенизатор не сломался и точно так же вытаскиваем лейбл этого отзыва, теперь же переходим к encoding самый важный момент, обращаемся к токенизатору и берем его оттуда и сохраняем его в нашу переменную,нам нужен сам текст,технические маячки(наподобии 101 и 102 для начала и конца текста), максимальная длина для того чтобы все матрицы были одного размера,отключим функцию return_token_type_ids которая нужна для задач подачи 2 текстов одновременно, затем padding и truncation которые будут подгонять под наш размер маленькие тексты и обрезать большие и поскольку padding добавляет кучу нулей мы сделаем маску вниманя через return_attention_mask чтобы берт смотрел только туда где есть 1,а 0 игнорировал ну и в конце нам нужно чтобы он отдал нам не просто список,а именно тензор торча. В конце нам нужно что бы нам выдало текст (для дальнейшей проверки), затем достаем наши закодированые слова (flatten нужен для того что бы он дал не матрицу,а строчку), достаем маску с таким же методом, в конце мы достаем наши лейблы которые превращаются в объект для pytorch и меняем их формат для удобной работы

In [5]:
import torch
from torch.utils.data import Dataset, DataLoader

class ReviewDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len 

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts.iloc[item])
        label = self.labels.iloc[item]

        encoding = self.tokenizer(
            text,
            add_special_tokens=True, 
            max_length=self.max_len, 
            return_token_type_ids=False,
            padding='max_length',   
            truncation=True,        
            return_attention_mask=True, 
            return_tensors='pt',     
        )

        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

фиксируем длину каждого отзыва и размер батча, затем создаем переменную по нашему классу, передаем внутрь текст,лейблы,токенизатор и определяем лимит токенов, то же самое для теста,но на другой табличке, затем создаем loader для трейна и теста где передаем размер батчей (в трейне указываем перемешивание для борьбы с переобучением,в тесте нам это не нужно), так же создаем загрузчик из валидации который мы будем использовать при обучении

In [6]:
BATCH_SIZE = 32


train_dataset = ReviewDataset(
    texts=dt_train['text'], 
    labels=dt_train['label'], 
    tokenizer=tokenizer, 
    max_len=MAX_LEN
)

test_dataset = ReviewDataset(
    texts=dt_test['text'], 
    labels=dt_test['label'], 
    tokenizer=tokenizer, 
    max_len=MAX_LEN
)


val_dataset = ReviewDataset(
    texts=dt_val['text'], 
    labels=dt_val['label'], 
    tokenizer=tokenizer, 
    max_len=MAX_LEN
)


val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

импортируем берт для классификации, скачиваем нашу модель и даем ей слой для классификации на 3 выхода который мы и будем обучать, ищем девайс (желательно cuda) и кидаем модель на девайс

In [7]:
from transformers import BertForSequenceClassification



model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=3,  
    ignore_mismatched_sizes=True
)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


model.to(device)

print(f"отправил на: {device}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you

отправил на: cuda


Сначала нам нужно проверить нашу модель из коробки для того что бы в конце понять улучшили мы ее или лучше просто взять стоковую, делаем модель в режим теста и заводим списки для оценки качества модели, выключаем запись градов, затем по батчам берем наши токены,маски и ответы и кидаем на девайс, делаем прямой проход, затем по логитам модель выдает классы и забираем ответы с видеокарты и переводим в формат нампай, затем сравниваем ответы модели с правильными и на этой основе с строим таблицу,видно,что результат стоковой модели довольно плох,по сути просто гадание

In [8]:
from sklearn.metrics import classification_report
import torch

model.eval()

predictions_before = []
true_labels_before = []


with torch.no_grad():
    
    for batch in tqdm(test_loader, desc="Тест ДО обучения"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        
        
        preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
        
        predictions_before.extend(preds)
        true_labels_before.extend(labels.cpu().numpy())

print("Модель до обучения:")
print(classification_report(true_labels_before, predictions_before, target_names=['Негатив', 'Нейтрально', 'Позитив']))


Тест ДО обучения: 100%|██████████| 47/47 [00:20<00:00,  2.29it/s]

Модель до обучения:
              precision    recall  f1-score   support

     Негатив       0.33      1.00      0.50       500
  Нейтрально       0.00      0.00      0.00       500
     Позитив       0.50      0.00      0.00       500

    accuracy                           0.33      1500
   macro avg       0.28      0.33      0.17      1500
weighted avg       0.28      0.33      0.17      1500




c:\Users\asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(avera

Теперь берем оптимизатор, выберем AdamW(обычный Адам,но который борется с переобучением,что с таким количествам параметров для нас идеально), прописываем эпохи и лр (лр берем максимально маленьким чтобы наша модель не подстроилась под наши отзывы), передаем все это в переменную вместе с параметрами и лр, затем создаем функцию контроля чтобы понимать не переобучилась ли наша модель, включаем eval потому что нам нужна только проверка,так что веса обновляться не должны, создаем счетчик лосс,with torch.no_grad(): нужна для ускорения тк при проверки нам не нужно запоминать и хранить грады, берем батчи и из них наши токены,масики и правильные ответы кидая их всех на видеокарту,затем проверяем прямым проходом наш лосс сравнивая правильные ответы с ответами модели, берем число из тензора который нам отдают и в конце мы возвращаем средний лосс по всем батчам

In [9]:
from torch.optim import AdamW

LEARNING_RATE = 3e-5

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

def evaluate_loss(model, dataloader):
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_loss += outputs.loss.item()
            
    return total_loss / len(dataloader)


Задаем настройки обучения,количество эпох, терпение модели(сколько эпох ждать перед прерыванием обучения если лосс не падает), счетчик эпох без улучшения, best_val_loss = float('inf') задаем для того что бы наша модель сохранилась даже при первой эпохе, затем блок обучения в котором мы переводим модель в режим обновления весов,заводим счетчик для наших лоссов, потом по батчам обучяем нашу модель беря наши токены,маски и ответы загоняя их на наш девайс, обнуляем град для того что бы не копилась ошибка и делаем прямой проход,считаем наш лосс и делаем обратное распространение с обновлением весов и считаем наш лосс по эпохе, затем включаем функцию мини экзамена и проверяем нашу модель, если ошибка меньше чем предыдущая то обновляем рекорд, затем обнуляем счетчик терпения и сохраняем нашу модель со всеми ее весами если же на этой эпохе ничего не улучшилось то модель учится дальше и добавляет +1 к счетчику (если мы дойдем до 2 то модель просто останвоится и у нас на руках будет модель с ее лучшим лосс за все время)    

In [10]:
EPOCHS = 10
patience = 2
patience_counter = 0
best_val_loss = float('inf')

for epoch in range(EPOCHS):
    print(f"\nЭпоха {epoch + 1} / {EPOCHS}")
    
    
    model.train() 
    total_train_loss = 0
    
    
    for batch in tqdm(train_loader, desc="Обучение"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        
        loss = outputs.loss
        total_train_loss += loss.item()
        loss.backward()
        optimizer.step()
        
    avg_train_loss = total_train_loss / len(train_loader)
    
   
    print("Проверка на валидационной выборке...")
    val_loss = evaluate_loss(model, val_loader)
    
    print(f"Train Loss: {avg_train_loss:.4f} | Val Loss: {val_loss:.4f}")
    
    
    if val_loss < best_val_loss:
        
        best_val_loss = val_loss
        patience_counter = 0 
        torch.save(model.state_dict(), 'best_model.pt')
        print("Найдена лучшая модель,веса сохранены в 'best_model.pt'.")
    else:
        
        patience_counter += 1
        print(f"Ошибка на валидации не упала. Счетчик терпения: {patience_counter} из {patience}")
        
    
    if patience_counter >= patience:
        print("Сработала ранняя остановка! Модель начала переобучаться.")
        break

print("\nОбучение завершено")


Эпоха 1 / 1


Обучение:  44%|████▍     | 145/329 [42:31<53:58, 17.60s/it]  


KeyboardInterrupt: 

импортируем нампай и классификатор ошибок, переводим модель в режим теста (ничего не обновляет просто фиксирует модель) и создаем пустые коробки для будущих отчетов, нам уже не нужны градиенты на тесте поэтому вырубаем их запись через with torch.no_grad() это сократит время на обучение и экономит память, точно так же берем батчи только уже по тестовой выборке, затем прямой проход и выдача сырых логитов, затем разбор сырого логита torch.argmax(logits, dim=1) берет самое большое число и его индекс (предсказание модели), затем .cpu() кидает все с видеокарты в оперативку и нампай превращает тензор пайторча в обычный массив, заполняем наши коробки кладя каждый батч ответы, с правильными ответами так же перекидываем в память и меняем формат, затем sklearn срванивает коробки с ответами модели и правильными овтетами и на их основе строит таблицу ошибок

In [ ]:
print("\nЗагружаем лучшие веса модели для финального тестирования")
model.load_state_dict(torch.load('best_model.pt'))

print("\nФинальный отчет о классификации")
model.eval()
predictions = []
true_labels = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Тестирование"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        
        predictions.extend(preds)
        true_labels.extend(labels.cpu().numpy())

from sklearn.metrics import classification_report
print(classification_report(true_labels, predictions, target_names=['Негатив', 'Нейтрально', 'Позитив']))


Загружаем лучшие веса модели для финального тестирования


FileNotFoundError: [Errno 2] No such file or directory: 'best_model.pt'